# nb01 — Séries do evento por ONDA (interativo)

Explora **Jz, raios (GLM), IVT, VPI e fumaça** dia a dia, com as faixas das ondas.
Escolha as variáveis e o intervalo com os controles. Rode as células de cima para baixo.

> Precisa de `viz_helpers.py`, `ondas_config.py` e da pasta `resultados/` na mesma pasta.

In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as W
from IPython.display import display
import importlib, ondas_config, viz_helpers as vh
importlib.reload(ondas_config); importlib.reload(vh)  # pega a versão mais nova sem reiniciar o kernel
from ondas_config import ONDAS
%matplotlib inline
df = vh.carrega_series('resultados')
assert df is not None, 'não achei resultados/tabela_diaria_integrada_2024.csv'
print('período:', df.datetime.min().date(),'->',df.datetime.max().date())
df.head()

## Faixas de onda + painéis por variável
Cada variável vira um painel separado (mais fácil de ler); as faixas coloridas marcam O0–O3.

In [ ]:
CORES_ONDA=['#999999','#D55E00','#CC79A7','#0072B2','#E69F00']
VARS={'Jz_rms':'Jz_rms (corrente vertical)','flashes':'Raios GLM/dia','IVT_kg_m_s':'IVT (umidade p/ RS)',
      'VPI_baixo_PVU':'VPI baixos níveis (PVU)','aod550_fumaca':'AOD fumaça'}
VARS={k:v for k,v in VARS.items() if k in df.columns}

def painel(variaveis, d0, d1, log_flashes=True):
    d=df[(df.datetime>=str(d0))&(df.datetime<=str(d1))]
    n=len(variaveis)
    if n==0: print('escolha ao menos 1 variável'); return
    fig,ax=plt.subplots(n,1,figsize=(11,2.6*n),sharex=True); ax=np.atleast_1d(ax)
    for i,v in enumerate(variaveis):
        for j,(nome,(i0,i1)) in enumerate(ONDAS.items()):
            ax[i].axvspan(pd.Timestamp(i0),pd.Timestamp(i1)+pd.Timedelta(hours=23),
                          color=CORES_ONDA[j%len(CORES_ONDA)],alpha=0.13)
        ax[i].plot(d.datetime,d[v],marker='o',ms=4,lw=1.5,color='#111')
        ax[i].set_title(VARS[v],loc='left',fontsize=11); ax[i].grid(alpha=0.3)
        if v=='flashes' and log_flashes: ax[i].set_yscale('symlog',linthresh=1000)
    for j,(nome,(i0,i1)) in enumerate(ONDAS.items()):
        xm=pd.Timestamp(i0)+(pd.Timestamp(i1)-pd.Timestamp(i0))/2
        ax[0].text(xm,ax[0].get_ylim()[1],nome.split('_')[0],ha='center',va='bottom',
                   fontsize=9,fontweight='bold',color=CORES_ONDA[j%len(CORES_ONDA)])
    plt.tight_layout(); plt.show()

sel=W.SelectMultiple(options=list(VARS), value=[k for k in ['IVT_kg_m_s','flashes','Jz_rms','aod550_fumaca'] if k in VARS],
                     description='variáveis', rows=6)
d0=W.DatePicker(description='de', value=df.datetime.min().date())
d1=W.DatePicker(description='até', value=df.datetime.max().date())
W.interact(painel, variaveis=sel, d0=d0, d1=d1, log_flashes=W.Checkbox(value=True,description='flashes em log'));

## Defasagem: os raios/Jz lideram ou seguem a umidade?
Correlação cruzada simples entre pares de variáveis (defasagem em dias).

In [ ]:
def xcorr(a,b,maxlag=4):
    a=(a-a.mean())/a.std(); b=(b-b.mean())/b.std(); n=len(a); out=[]
    for k in range(-maxlag,maxlag+1):
        if k<0: r=np.corrcoef(a[-k:],b[:n+k])[0,1]
        elif k>0: r=np.corrcoef(a[:n-k],b[k:])[0,1]
        else: r=np.corrcoef(a,b)[0,1]
        out.append((k,r))
    return out

def ver_xcorr(va,vb):
    d=df.dropna(subset=[va,vb])
    xs=xcorr(d[va].values,d[vb].values)
    k=[x[0] for x in xs]; r=[x[1] for x in xs]
    plt.figure(figsize=(7,3)); plt.bar(k,r,color='#0072B2'); plt.axhline(0,color='k',lw=.6)
    plt.xlabel(f'defasagem (dias)  —  {va} vs {vb}  (k>0: {va} lidera)'); plt.ylabel('correlação'); plt.grid(alpha=.3); plt.show()
opts=[k for k in VARS]
W.interact(ver_xcorr, va=W.Dropdown(options=opts,value=opts[0]), vb=W.Dropdown(options=opts,value=opts[-1]));